# Phase 3: Tuned XGBoost (improved)

Supervised flow-level classifier on the fixed Phase 3 splits.

**Changes vs the first XGBoost pass**
- Search `scale_pos_weight` instead of fixing it to `n_neg/n_pos` (~24), which pushed recall high and precision/F1 down.
- More `RandomizedSearchCV` trials.
- Choose the classification threshold on the **validation** set to maximize F1 (not a hard 0.5).
- Compare against the Random Forest baseline (~0.75 test F1).


## Setup and load the fixed flow splits

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve, make_scorer
)

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data.flow_preprocessing import ATTACK_COL, BINARY_COL, FLOW_KEY_COL, TARGET_COL, build_flow_preprocessor

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "phase3"
with open(DATA_DIR / "flow_split_metadata.json") as f:
    metadata = json.load(f)

train_df = pd.read_csv(DATA_DIR / "flow_train.csv", low_memory=False)
validation_df = pd.read_csv(DATA_DIR / "flow_validation.csv", low_memory=False)
test_df = pd.read_csv(DATA_DIR / "flow_test.csv", low_memory=False)

print(train_df.shape, validation_df.shape, test_df.shape)
metadata["split_rows"]

### Verify splits

In [ ]:
for name, current_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(name, current_df[BINARY_COL].value_counts().to_dict())

train_keys = set(train_df[FLOW_KEY_COL])
print("flow-key overlaps:", {
    "train∩val": len(train_keys & set(validation_df[FLOW_KEY_COL])),
    "train∩test": len(train_keys & set(test_df[FLOW_KEY_COL])),
    "val∩test": len(set(validation_df[FLOW_KEY_COL]) & set(test_df[FLOW_KEY_COL])),
})

## Features

In [ ]:
feature_cols = metadata["feature_columns"]
missing = [c for c in feature_cols if c not in train_df.columns]
if missing:
    raise ValueError(missing)

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].to_numpy()
X_validation = validation_df[feature_cols]
y_validation = validation_df[TARGET_COL].to_numpy()
X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL].to_numpy()

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
full_spw = neg / max(pos, 1)
sqrt_spw = float(np.sqrt(full_spw))
print(f"neg={neg}, pos={pos}, full_spw={full_spw:.3f}, sqrt_spw={sqrt_spw:.3f}")
print("features:", len(feature_cols))

## Tune XGBoost

`scale_pos_weight` is searched over milder imbalance corrections. Full `n_neg/n_pos` often over-emphasizes recall and hurts F1 at a 0.5 threshold.


In [ ]:
RANDOM_STATE = 1
N_ITER = 24

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
preprocessor = build_flow_preprocessor(scale=False)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model),
])

param_distributions = {
    "model__n_estimators": [200, 300, 500, 700, 900],
    "model__max_depth": [3, 4, 6, 8, 10],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.5, 0.7, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "model__gamma": [0.0, 0.1, 0.5, 1.0],
    # milder than full imbalance ratio — RF baseline did not need extreme reweighting
    "model__scale_pos_weight": [1.0, 2.0, 3.0, 5.0, 8.0, sqrt_spw, min(full_spw, 15.0)],
}

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=False,
)

start_search = time.perf_counter()
xgb_search.fit(X_train, y_train)
search_time = time.perf_counter() - start_search

xgb_pipeline = xgb_search.best_estimator_
xgb_pipeline.named_steps["model"].set_params(n_jobs=-1)

print(f"Search time: {search_time:.1f}s")
print(f"Best CV PR-AUC: {xgb_search.best_score_:.4f}")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")

### Top CV configs

In [ ]:
cv_results_df = pd.DataFrame(xgb_search.cv_results_)
cols = [
    "rank_test_pr_auc", "mean_test_pr_auc", "mean_test_roc_auc", "mean_test_f1",
    "mean_test_precision", "mean_test_recall", "mean_fit_time",
    "param_model__n_estimators", "param_model__max_depth", "param_model__learning_rate",
    "param_model__scale_pos_weight", "param_model__subsample", "param_model__colsample_bytree",
]
display(cv_results_df[cols].sort_values("rank_test_pr_auc").head(10).round(4))

## Validation threshold search

With class imbalance, 0.5 is rarely optimal for F1. We sweep thresholds on validation probabilities and freeze the best F1 threshold for test.


In [ ]:
def evaluate_binary_classifier(name, y_true, scores, threshold, prediction_time):
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "dataset": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "fnr": fn / (fn + tp) if (fn + tp) else 0.0,
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "alerts": int(y_pred.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "prediction_time_sec": prediction_time,
    }, y_pred


def best_f1_threshold(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": 0.5, "f1": -1.0}
    rows = []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        rows.append({
            "threshold": float(t),
            "f1": f1,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "fpr": ((pred == 1) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1),
        })
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1)}
    return best, pd.DataFrame(rows)

start = time.perf_counter()
validation_scores = xgb_pipeline.predict_proba(X_validation)[:, 1]
validation_prediction_time = time.perf_counter() - start

threshold_info, threshold_curve = best_f1_threshold(y_validation, validation_scores)
CLASSIFICATION_THRESHOLD = threshold_info["threshold"]
print(f"Chosen validation threshold: {CLASSIFICATION_THRESHOLD:.4f} (val F1={threshold_info['f1']:.4f})")

# also report 0.5 for reference
ref_05, _ = evaluate_binary_classifier("Validation@0.5", y_validation, validation_scores, 0.5, validation_prediction_time)
print("Validation at 0.5:", {k: round(ref_05[k], 4) for k in ["precision", "recall", "f1"]})

plt.figure(figsize=(7, 4))
plt.plot(threshold_curve["threshold"], threshold_curve["f1"], label="F1")
plt.plot(threshold_curve["threshold"], threshold_curve["precision"], label="Precision", alpha=0.8)
plt.plot(threshold_curve["threshold"], threshold_curve["recall"], label="Recall", alpha=0.8)
plt.axvline(CLASSIFICATION_THRESHOLD, linestyle="--", color="black", label="chosen")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("XGBoost validation threshold sweep")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluate on validation and test

In [ ]:
start = time.perf_counter()
test_scores = xgb_pipeline.predict_proba(X_test)[:, 1]
test_prediction_time = time.perf_counter() - start

validation_metrics, validation_pred = evaluate_binary_classifier(
    "Validation", y_validation, validation_scores, CLASSIFICATION_THRESHOLD, validation_prediction_time
)
test_metrics, test_pred = evaluate_binary_classifier(
    "Test", y_test, test_scores, CLASSIFICATION_THRESHOLD, test_prediction_time
)
metrics_df = pd.DataFrame([validation_metrics, test_metrics])
display(metrics_df.round(4))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], ["Benign", "Attack"])
    plt.yticks([0, 1], ["Benign", "Attack"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_validation, validation_pred, "XGBoost Validation CM")
plot_confusion_matrix(y_test, test_pred, "XGBoost Test CM")

In [ ]:
report_df = pd.DataFrame(classification_report(
    y_test, test_pred, target_names=["Benign", "Attack"], output_dict=True, zero_division=0
)).T
display(report_df.round(4))

type_results = test_df[[ATTACK_COL]].copy()
type_results["y_true"] = y_test
type_results["y_pred"] = test_pred
type_results["correct"] = type_results["y_true"] == type_results["y_pred"]
result_by_type = type_results.groupby(ATTACK_COL).agg(total=("correct", "size"), correct=("correct", "sum"))
result_by_type["correct_rate"] = result_by_type["correct"] / result_by_type["total"] * 100
type_order = ["Benign", "DDoS-HTTP Flood", "DoS-HTTP Flood", "DNS Spoofing", "Brute Force", "XSS"]
result_by_type = result_by_type.reindex(type_order).dropna()
display(result_by_type.round(2))

In [ ]:
def plot_roc_curve(y_true, scores, title):
    fpr, tpr, _ = roc_curve(y_true, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC-AUC={roc_auc_score(y_true, scores):.4f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(title); plt.legend(); plt.grid(True); plt.show()

def plot_precision_recall_curve(y_true, scores, title):
    p, r, _ = precision_recall_curve(y_true, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(r, p, label=f"PR-AUC={average_precision_score(y_true, scores):.4f}")
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(title); plt.legend(); plt.grid(True); plt.show()

plot_roc_curve(y_test, test_scores, "XGBoost Test ROC")
plot_precision_recall_curve(y_test, test_scores, "XGBoost Test PR")

## Feature importance and save

In [ ]:
processed_feature_names = xgb_pipeline.named_steps["preprocessor"].get_feature_names_out(feature_cols)
importance_df = pd.DataFrame({
    "feature": processed_feature_names,
    "importance": xgb_pipeline.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
display(importance_df.head(20))

plot_importance = importance_df.head(20).sort_values("importance")
plt.figure(figsize=(9, 7))
plt.barh(plot_importance["feature"], plot_importance["importance"])
plt.title("Top 20 XGBoost features")
plt.tight_layout()
plt.show()

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "reports" / "phase3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

metrics_output = metrics_df.copy()
metrics_output.insert(0, "model", "XGBoost Tuned Improved")
metrics_output["search_time_sec"] = search_time
metrics_output["refit_time_sec"] = xgb_search.refit_time_
metrics_output.to_csv(RESULTS_DIR / "xgboost_tuned_metrics.csv", index=False)
importance_df.to_csv(RESULTS_DIR / "xgboost_tuned_feature_importance.csv", index=False)
cv_results_df.to_csv(RESULTS_DIR / "xgboost_random_search_results.csv", index=False)

best_parameters = {
    "best_cv_pr_auc": float(xgb_search.best_score_),
    "search_time_sec": search_time,
    "refit_time_sec": xgb_search.refit_time_,
    "n_iter": N_ITER,
    "cv_folds": cv.get_n_splits(),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "validation_threshold_f1": threshold_info["f1"],
    "best_params": {k: (float(v) if hasattr(v, "item") else v) for k, v in xgb_search.best_params_.items()},
}
(RESULTS_DIR / "xgboost_best_parameters.json").write_text(json.dumps(best_parameters, indent=2))

pred = test_df[[FLOW_KEY_COL, ATTACK_COL, BINARY_COL, TARGET_COL]].copy()
pred["attack_probability"] = test_scores
pred["prediction"] = test_pred
pred.to_csv(RESULTS_DIR / "xgboost_tuned_test_predictions.csv", index=False)
print("Saved to", RESULTS_DIR)

In [ ]:
test_row = metrics_df.loc[metrics_df["dataset"] == "Test"].iloc[0]
print("Final improved XGBoost test results")
print("----------------------------------")
print(f"Best CV PR-AUC: {xgb_search.best_score_:.4f}")
print(f"Threshold: {CLASSIFICATION_THRESHOLD:.4f}")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "fpr", "fnr"]:
    print(f"{k}: {test_row[k]:.4f}")
print("Best params:")
for k, v in xgb_search.best_params_.items():
    print(f"  {k}: {v}")